# Silver Transformation — London Weather

Transform raw Open-Meteo weather snapshots into validated, analytics-ready observations.

This notebook:

1. Reads Bronze weather snapshots.
2. Parses JSON using an explicit schema.
3. Normalizes field names and units.
4. Converts local weather time to UTC.
5. Adds weather-code descriptions.
6. Applies data-quality checks.
7. Removes duplicate observations.
8. Merges validated data into Silver.

**Source:** `workspace.urbanpulse_bronze.weather`

**Target:** `workspace.urbanpulse_silver.weather`

## 1. Initialise project paths

In [0]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
SRC_PATH = PROJECT_ROOT / "src"

if not SRC_PATH.exists():
    raise FileNotFoundError(
        f"Source directory not found: {SRC_PATH}"
    )

if str(SRC_PATH) not in sys.path:
    sys.path.insert(
        0,
        str(SRC_PATH),
    )

print(f"Project root: {PROJECT_ROOT}")

## 2. Import transformation components

In [0]:
from pyspark.sql import functions as F

from urbanpulse.transformations.weather import (
    transform_weather,
)

from urbanpulse.quality.weather import (
    valid_weather,
    invalid_weather,
)

from urbanpulse.utils.delta import (
    merge_insert_only,
)

## 3. Define source and target tables

In [0]:
BRONZE_TABLE = (
    "workspace."
    "urbanpulse_bronze."
    "weather"
)

SILVER_TABLE = (
    "workspace."
    "urbanpulse_silver."
    "weather"
)

## 4. Read Bronze weather snapshots

In [0]:
bronze_df = spark.table(
    BRONZE_TABLE
)

print(
    f"Bronze weather snapshots: "
    f"{bronze_df.count()}"
)

display(
    bronze_df.select(
        "request_id",
        "ingested_at",
        "http_status",
    )
    .orderBy(
        F.col("ingested_at").desc()
    )
)

## 5. Parse and normalize weather observations

Parse the Open-Meteo response using the explicit weather schema and convert source fields into consistent analytical names.

In [0]:
parsed_df = transform_weather(
    bronze_df
)

print(
    f"Parsed observations: "
    f"{parsed_df.count()}"
)

display(parsed_df)

## 6. Verify weather timestamps

Open-Meteo returns London-local observation time.

UrbanPulse also derives a UTC timestamp for consistent joins with transport datasets.

In [0]:
display(
    parsed_df.select(
        "weather_observed_at_local",
        "weather_timezone",
        "utc_offset_seconds",
        "weather_observed_at",
        "snapshot_at",
    )
    .orderBy(
        F.col(
            "weather_observed_at"
        ).desc()
    )
)

## 7. Inspect normalized weather measurements

In [0]:
display(
    parsed_df.select(
        "temperature_c",
        "relative_humidity_pct",
        "precipitation_mm",
        "rain_mm",
        "weather_code",
        "weather_description",
        "cloud_cover_pct",
        "wind_speed_kmh",
        "wind_gusts_kmh",
        "weather_observed_at",
    )
)

## 8. Apply data-quality rules

Weather observations must have:

- valid coordinates
- observation timestamp
- humidity between 0–100%
- cloud cover between 0–100%
- non-negative precipitation
- non-negative rainfall
- non-negative wind measurements
- ingestion timestamp

In [0]:
valid_df = valid_weather(
    parsed_df
)

invalid_df = invalid_weather(
    parsed_df
)

valid_count = valid_df.count()
invalid_count = invalid_df.count()

print(
    f"Valid observations: {valid_count}"
)

print(
    f"Invalid observations: {invalid_count}"
)

## 9. Enforce the weather data contract

Unexpected invalid weather observations stop the Silver transformation so source or schema changes are not silently ignored.

In [0]:
if invalid_count > 0:
    display(invalid_df)

    raise ValueError(
        f"Data quality failure: "
        f"{invalid_count} invalid "
        "weather observations"
    )

print(
    "Weather quality checks passed."
)

## 10. Create the weather observation key

A deterministic SHA-256 key uniquely identifies a weather observation for a location and observation timestamp.

In [0]:
keyed_df = (
    valid_df
    .withColumn(
        "weather_observation_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.col(
                    "latitude"
                ).cast("string"),
                F.col(
                    "longitude"
                ).cast("string"),
                F.col(
                    "weather_observed_at"
                ).cast("string"),
            ),
            256,
        ),
    )
)

## 11. Deduplicate weather observations

Multiple API requests can return the same weather observation timestamp.

Silver retains one observation per location and weather timestamp.

In [0]:
deduplicated_df = (
    keyed_df
    .orderBy(
        F.col("snapshot_at").desc()
    )
    .dropDuplicates([
        "weather_observation_key"
    ])
)

print(
    f"Rows before deduplication: "
    f"{valid_count}"
)

print(
    f"Rows after deduplication: "
    f"{deduplicated_df.count()}"
)

## 12. Add Silver processing metadata

In [0]:
silver_df = (
    deduplicated_df
    .withColumn(
        "processed_at",
        F.current_timestamp(),
    )
)

## 13. Verify weather observation keys

In [0]:
duplicate_keys_df = (
    silver_df
    .groupBy(
        "weather_observation_key"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

if duplicate_keys_df.count() > 0:
    display(duplicate_keys_df)

    raise ValueError(
        "Duplicate weather observation "
        "keys detected."
    )

print(
    "Weather observation keys are unique."
)

## 14. Merge observations into Silver

Delta `MERGE` ensures rerunning the transformation does not duplicate existing weather observations.

In [0]:
merge_result = merge_insert_only(
    spark=spark,
    source_df=silver_df,
    target_table=SILVER_TABLE,
    merge_condition="""
        target.weather_observation_key
        =
        source.weather_observation_key
    """,
)

print(
    f"Silver weather table: "
    f"{merge_result}"
)

## 15. Verify Silver weather observations

In [0]:
%sql
SELECT
    weather_observation_key,
    weather_observed_at,
    temperature_c,
    relative_humidity_pct,
    precipitation_mm,
    rain_mm,
    weather_code,
    weather_description,
    cloud_cover_pct,
    wind_speed_kmh,
    wind_gusts_kmh,
    snapshot_at,
    processed_at
FROM workspace.urbanpulse_silver.weather
ORDER BY weather_observed_at DESC;

In [0]:
%sql
SELECT
    COUNT(*) AS observations,
    COUNT(DISTINCT weather_observation_key) AS unique_observations,
    MIN(weather_observed_at) AS earliest_observation,
    MAX(weather_observed_at) AS latest_observation
FROM workspace.urbanpulse_silver.weather;

In [0]:
%sql
SELECT *
FROM workspace.urbanpulse_silver.weather
WHERE
    relative_humidity_pct NOT BETWEEN 0 AND 100
    OR cloud_cover_pct NOT BETWEEN 0 AND 100
    OR precipitation_mm < 0
    OR rain_mm < 0
    OR wind_speed_kmh < 0
    OR wind_gusts_kmh < 0;

In [0]:
%sql
SELECT
    weather_observation_key,
    COUNT(*) AS records
FROM workspace.urbanpulse_silver.weather
GROUP BY weather_observation_key
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT COUNT(*)
FROM workspace.urbanpulse_silver.weather;

In [0]:
%sql
SELECT
    weather_observed_at,
    temperature_c,
    weather_description
FROM workspace.urbanpulse_silver.weather
ORDER BY weather_observed_at DESC;